# Filter Audition

Loads `static_10m_000.wav`, applies filters, and lets you listen to **raw vs filtered** side-by-side.  
Pick a channel to monitor — outer ring channels 1–4 recommended.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Audio

from logic.wav_loader import WavLoader
from logic.filters import HighPassFilter, PeakingFilter

%matplotlib inline


In [ ]:
# --- Config ---
WAV_FILE     = '../data/static_10m_000.wav'
SAMPLE_RATE  = 44100
CHUNK_SIZE   = 8192
MONITOR_CH   = 0        # channel index to listen to (0 = CH1, 1 = CH2, ...)

# Filter params
HP_CUTOFF_HZ = 400
PEAK_FREQ_HZ = 1009
PEAK_GAIN_DB = 20.0
PEAK_Q       = 0.71


In [ ]:
def collect_audio(apply_filters: bool) -> np.ndarray:
    """Run the full file through the pipeline and return (n_samples, n_channels) float32."""
    loader = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
    stream = loader.stream()
    if apply_filters:
        hp   = HighPassFilter(cutoff_hz=HP_CUTOFF_HZ, sampling_rate=SAMPLE_RATE, order=4)
        peak = PeakingFilter(center_hz=PEAK_FREQ_HZ, gain_db=PEAK_GAIN_DB, q=PEAK_Q, sampling_rate=SAMPLE_RATE)
        stream = peak.process(hp.process(stream))
    return np.concatenate([chunk.data for chunk in stream], axis=0)

raw      = collect_audio(apply_filters=False)
filtered = collect_audio(apply_filters=True)

print(f'Raw:      shape={raw.shape}  max={raw.max():.4f}  min={raw.min():.4f}')
print(f'Filtered: shape={filtered.shape}  max={filtered.max():.4f}  min={filtered.min():.4f}')


In [ ]:
# --- Waveform comparison ---
t = np.arange(len(raw)) / SAMPLE_RATE

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(t, raw[:, MONITOR_CH], linewidth=0.5)
axes[0].set_title(f'Raw — CH{MONITOR_CH + 1}', fontsize=11)
axes[0].set_ylabel('Amplitude')

axes[1].plot(t, filtered[:, MONITOR_CH], linewidth=0.5, color='tomato')
axes[1].set_title(f'Filtered (HP {HP_CUTOFF_HZ}Hz + Peak {PEAK_FREQ_HZ}Hz +{PEAK_GAIN_DB:.0f}dB) — CH{MONITOR_CH + 1}', fontsize=11)
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()


In [ ]:
# --- Listen: Raw ---
print(f'▶ Raw — CH{MONITOR_CH + 1}')
display(Audio(raw[:, MONITOR_CH], rate=SAMPLE_RATE, normalize=False))


In [ ]:
# --- Listen: Filtered ---
print(f'▶ Filtered — CH{MONITOR_CH + 1}')
display(Audio(filtered[:, MONITOR_CH], rate=SAMPLE_RATE, normalize=False))
